In [1]:
#importamos las librerías necesarias para la exploración inicial

from pathlib import Path
import pandas as pd
import numpy as np
import json

In [2]:
# Preparamos rutas, localizamos archivos y mostramos sus características

#dataset original
DATA_DIR = Path("../Data/RAW")

#dataset limpio
OUTPUT_DIR = Path("../Data/Clean")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Archivos CSV encontrados: {len(csv_files)}\n")

for file in csv_files:
    print(f"{file.name:40} {file.stat().st_size / (1024**2):8.2f} MB")

Archivos CSV encontrados: 11

hourly_weather.csv                           2.78 MB
Ing_in_item.csv                              0.00 MB
Ingredient.csv                               0.01 MB
Item_on_receipt.csv                         56.48 MB
Menu_item.csv                                0.03 MB
Payment.csv                                 22.50 MB
Receipt.csv                                 68.09 MB
SA1_AppWichStoreAttributes.csv               0.00 MB
SA1_county_crime.csv                         0.00 MB
SA1_county_demographic.csv                   0.00 MB
SA1_county_employment.csv                    0.00 MB


In [3]:
#cargamos archivos CSV, comprobamos su lectura
#y dfinimos el tratamiento inicial de valores nulos

datasets = {}

for file in csv_files:
    try:
        df = pd.read_csv(
            file,
            encoding = "utf-8-sig",
            low_memory=False,
            keep_default_na=False,
            na_values=[""]
        )

        datasets[file.stem] = df

        print(f"{file.name}: "
             f"{df.shape[0]}"
             )
    except Exception as e:
        print (f"Error leyendo {file.name}: {e}")

hourly_weather.csv: 42909
Ing_in_item.csv: 78
Ingredient.csv: 279
Item_on_receipt.csv: 1048575
Menu_item.csv: 560
Payment.csv: 580453
Receipt.csv: 575507
SA1_AppWichStoreAttributes.csv: 10
SA1_county_crime.csv: 6
SA1_county_demographic.csv: 6
SA1_county_employment.csv: 6


In [4]:
#visualizamos el contenido de cada tabla
#y guardamos la información en un dataframe de pandas

inventory = []

for name, df in datasets.items():
    inventory.append({
        "tabla":name,
        "filas":len(df),
        "columnas":len(df.columns),
        "columnas_lista":", ".join(map(str,df.columns))
    })

inventory_df = pd.DataFrame(inventory)

#mostramos la información utilizando la función display
display(
    inventory_df.sort_values("tabla").reset_index(drop=True)
)



,tabla,filas,columnas,columnas_lista
0,Ing_in_item,78,5,"Ingredient_ID, Inv_Num, Quantity, Unit, Unit_Amt"
1,Ingredient,279,2,"Ingredient_ID, Description"
2,Item_on_receipt,1048575,7,"Receipt_ID, Inv_Num, Price, Food_Cost, Count, ..."
3,Menu_item,560,7,"Inv_Num, Item_Description, Size, Cat_ID, Cat_D..."
4,Payment,580453,4,"Receipt_ID, Payment_ID, Pay_Type, Tender_Amt"
5,Receipt,575507,17,"Receipt_ID, Store_Num, Date, Start_Time, End_T..."
6,SA1_AppWichStoreAttributes,10,14,"Store_Name, Store_Num, Store_City, Store_Count..."
7,SA1_county_crime,6,12,"County_Name, County_Crime_Pop, County_Total_Cr..."
8,SA1_county_demographic,6,14,"County_Name, County_Total_Census_Pop, County_N..."
9,SA1_county_employment,6,5,"County_Name, County_Labor_Force, County_Employ..."


In [6]:
#Extraemos una muestra de los valores de las tablas

for name, df in datasets.items():
    print("="*20)
    print(f"Tabla: {name}")
    display(df.head(5))
    display(df.dtypes.rename("tipo actual").to_frame())
    print("*"*20)

Tabla: hourly_weather


,Weather_Station,Date,Hour,Air_Temp,Dew_Point,Pressure,Wind_Dir,Wind_Spd,Sky_Cond,Precip_1hr,Precip_6hr
0,River City Airport,4/1/12,0,44.96,35.96,29.529884,160.0,4.6977,0.0,0.0,NaN
1,River City Airport,4/1/12,1,46.94,32.00,29.535790,190.0,10.2902,0.0,0.0,NaN
2,River City Airport,4/1/12,2,48.92,33.08,29.547602,180.0,9.1717,8.0,0.0,NaN
3,River City Airport,4/1/12,3,50.00,32.00,29.574179,180.0,13.8694,8.0,0.0,NaN
4,River City Airport,4/1/12,4,48.92,32.00,29.627333,220.0,16.1064,8.0,-1.0,NaN


,tipo actual
Weather_Station,object
Date,object
Hour,int64
Air_Temp,float64
Dew_Point,float64
Pressure,float64
Wind_Dir,float64
Wind_Spd,float64
Sky_Cond,float64
Precip_1hr,float64


********************
Tabla: Ing_in_item


,Ingredient_ID,Inv_Num,Quantity,Unit,Unit_Amt
0,1501,11291,1,EACH,1
1,1621,11291,1,EACH,1
2,1641,11291,1,EACH,1
3,1661,11291,1,EACH,1
4,13751,11291,1,EACH,1


...

,tipo actual
County_Name,object
County_Total_Census_Pop,int64
County_Non-Hispanic_White,int64
County_Non-Hispanic_Black,int64
County_Non-Hispanic_Native_American,int64
County_Non-Hispanic_Asian,int64
County_Non-Hispanic_Pacific_Islander,int64
County_Non-Hispanic_Two_or_more,int64
County_Hispanic_White,int64
County_Hispanic_Black,int64


********************
Tabla: SA1_county_employment


,County_Name,County_Labor_Force,County_Employed,County_Unemployed,County_Unemployment_Rate
0,Appleton County,"121,082","108,267","12,815",10.6
1,Farm County,"36,674","33,354","3,320",9.1
2,Lake County,"43,914","39,740","4,174",9.5
3,Power County,"92,041","83,467","8,574",9.3
4,Rail County,"39,167","35,366","3,801",9.7


,tipo actual
County_Name,object
County_Labor_Force,object
County_Employed,object
County_Unemployed,object
County_Unemployment_Rate,float64


********************
